# Multi-Temporal Analysis with DINOv3 Embeddings
**PyGeoVision v2.0 — Notebook 24**

## Real-World Problem
Analysts need to find all scenes in a 10,000-scene archive that are
similar to a flooding event — without manually reviewing each scene.
DINOv3 embeddings enable semantic similarity search in seconds.

## Learning Objectives
1. Extract DINOv3 CLS embeddings for satellite image retrieval
2. Build a FAISS similarity search index
3. Detect anomalies in time series using embedding distances
4. Cluster scenes unsupervised
5. Visualise embedding space with PCA and t-SNE

```bash
pip install "pygeovision[geo,foundation]" scikit-learn
```

In [ ]:
import pygeovision as pgv
import numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
import warnings; warnings.filterwarnings('ignore')

BASE_DIR = Path("./outputs/24_dinov3_embeddings")
BASE_DIR.mkdir(parents=True, exist_ok=True)

BBOX   = (-74.10, 40.60, -73.70, 40.90)   # New York City
client = pgv.PyGeoVision()
print(client)
print("\nGoal: DINOv3 embedding-based semantic search and anomaly detection")

## Step 1: Build Monthly Time Series

In [ ]:
MONTHS = ["2024-01","2024-03","2024-05","2024-07","2024-09","2024-11"]
scenes = {}

print("Acquiring 6 monthly scenes...")
import calendar
for month in MONTHS:
    yr, mo   = month.split("-")
    last_day = calendar.monthrange(int(yr), int(mo))[1]
    results  = client.search(
        bbox=BBOX, date_range=(f"{month}-01",f"{month}-{last_day}"),
        providers=["planetary_computer"], cloud_cover_max=20,
        sort_by="cloud_cover", limit=1,
    )
    if results:
        dl = client.download(results[:1], output_dir=BASE_DIR/month.replace("-","_"),
                              post_process=["reproject:EPSG:32618","cog"])
        if dl and dl[0].success:
            scenes[month] = dl[0].path
    print(f"  {month}: {'found' if month in scenes else 'missing'}")
print(f"\nScenes: {len(scenes)}/{len(MONTHS)}")

## Step 2: Extract DINOv3 Embeddings

In [ ]:
from pygeovision.ai.geoai.dinov3_proxy import DINOv3Proxy
from pygeovision.models.foundation.dinov3 import DINOv3Backbone

proxy = DINOv3Proxy()
print("DINOv3 Embedding Extraction:")
print(f"  Model : dinov3_vitl16_sat (300M params)")
print(f"  Output: 1024-dimensional CLS token per scene")
print()

embeddings = {}
for month, path in scenes.items():
    if path and Path(path).exists():
        emb = proxy.extract_embeddings(path)   # (1, 1024)
        embeddings[month] = emb
        print(f"  {month}: shape={emb.shape}  norm={np.linalg.norm(emb):.3f}")
    else:
        # Seasonal synthetic embeddings
        np.random.seed(MONTHS.index(month))
        idx  = MONTHS.index(month)
        t    = 2*np.pi * idx / len(MONTHS)
        base = np.random.randn(1024) * 0.3
        seasonal = np.zeros(1024)
        seasonal[:256]  = 0.4*np.sin(t)    # Vegetation seasonal signal
        seasonal[256:512] = 0.2*np.cos(t)  # Water/snow signal
        embeddings[month] = (base + seasonal).reshape(1, 1024)
        print(f"  {month}: [synthetic]  norm={np.linalg.norm(embeddings[month]):.3f}")

print(f"\nEmbedding matrix: {len(embeddings)} scenes x 1024 dims")

## Step 3: Similarity Search

In [ ]:
month_list = list(embeddings.keys())
emb_matrix = np.vstack([embeddings[m] for m in month_list])   # (N, 1024)
emb_norm   = emb_matrix / (np.linalg.norm(emb_matrix, axis=1, keepdims=True) + 1e-8)

# Cosine similarity matrix
sim_matrix = cosine_similarity(emb_norm)

print("Embedding Similarity Matrix (cosine):")
print(f"{'':>8}", end="")
for m in month_list:
    print(f"  {m[5:]:>7}", end="")
print()
print("  " + "─" * (len(month_list)*9+5))

for i, mi in enumerate(month_list):
    print(f"  {mi[5:]:>6}", end="")
    for j in range(len(month_list)):
        val   = sim_matrix[i,j]
        color = "HIGH" if val > 0.9 and i != j else ""
        print(f"  {val:.4f}", end="")
    print()

print()
# Find most similar pair
max_sim = 0; best_pair = ("","")
for i in range(len(month_list)):
    for j in range(i+1,len(month_list)):
        if sim_matrix[i,j] > max_sim:
            max_sim = sim_matrix[i,j]; best_pair = (month_list[i],month_list[j])
print(f"Most similar pair: {best_pair[0]} <-> {best_pair[1]}  (sim={max_sim:.4f})")

# Find least similar pair
min_sim = 1; worst_pair = ("","")
for i in range(len(month_list)):
    for j in range(i+1,len(month_list)):
        if sim_matrix[i,j] < min_sim:
            min_sim = sim_matrix[i,j]; worst_pair = (month_list[i],month_list[j])
print(f"Least similar pair: {worst_pair[0]} <-> {worst_pair[1]}  (sim={min_sim:.4f})")

## Step 4: Anomaly Detection

In [ ]:
# Detect anomalous scenes using embedding distance from cluster centroid
centroid = emb_norm.mean(axis=0)
distances = np.array([1 - np.dot(emb_norm[i], centroid) for i in range(len(month_list))])

mean_dist = distances.mean()
std_dist  = distances.std()
z_scores  = (distances - mean_dist) / (std_dist + 1e-8)

print("Anomaly Detection (embedding distance from centroid):")
print()
print(f"{'Month':<10} {'Distance':>10} {'Z-score':>10} {'Status'}")
print("─" * 42)
for month, dist, z in zip(month_list, distances, z_scores):
    status = "ANOMALY" if abs(z) > 1.5 else ("unusual" if abs(z) > 1.0 else "normal")
    flag   = " <--" if abs(z) > 1.5 else ""
    print(f"  {month:<8}  {dist:>9.4f}  {z:>9.2f}  {status}{flag}")

anomalies = [m for m, z in zip(month_list, z_scores) if abs(z) > 1.5]
print(f"\nAnomalous scenes: {anomalies or 'none'}")
print("Interpretation:")
print("  High distance from centroid -> unusual season/event (flood, snowstorm, etc.)")

## Step 5: PCA and t-SNE Visualisation

In [ ]:
# PCA
pca    = PCA(n_components=3)
emb_2d = pca.fit_transform(emb_norm)

print(f"PCA variance explained: {pca.explained_variance_ratio_.sum():.1%}")
print(f"  PC1: {pca.explained_variance_ratio_[0]:.1%}")
print(f"  PC2: {pca.explained_variance_ratio_[1]:.1%}")
print(f"  PC3: {pca.explained_variance_ratio_[2]:.1%}")
print()
print("PC loadings interpretation:")
print("  PC1 likely captures: seasonal NDVI (summer-winter contrast)")
print("  PC2 likely captures: moisture/cloud variability")
print("  PC3 likely captures: sensor/atmospheric noise")

month_colors = ['#1F77B4','#2CA02C','#FF7F0E','#D62728','#9467BD','#8C564B']

## Step 6: Full Embedding Dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# PCA scatter
for i, (month, color) in enumerate(zip(month_list, month_colors)):
    axes[0,0].scatter(emb_2d[i,0], emb_2d[i,1], c=color, s=250, zorder=5,
                       edgecolor='black', linewidth=1.2)
    axes[0,0].annotate(month, (emb_2d[i,0],emb_2d[i,1]),
                        textcoords="offset points", xytext=(8,5), fontsize=10, fontweight='bold')
axes[0,0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%})")
axes[0,0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%})")
axes[0,0].set_title("DINOv3 Embeddings in 2D
(PCA projection)", fontsize=11, fontweight='bold')
axes[0,0].grid(True, alpha=0.3)

# Similarity matrix heatmap
im = axes[0,1].imshow(sim_matrix, cmap='RdYlGn', vmin=0.5, vmax=1.0)
axes[0,1].set_xticks(range(len(month_list))); axes[0,1].set_xticklabels([m[5:] for m in month_list], rotation=45)
axes[0,1].set_yticks(range(len(month_list))); axes[0,1].set_yticklabels([m[5:] for m in month_list])
plt.colorbar(im, ax=axes[0,1], fraction=0.046, pad=0.04)
for i in range(len(month_list)):
    for j in range(len(month_list)):
        axes[0,1].text(j, i, f"{sim_matrix[i,j]:.3f}", ha='center', va='center', fontsize=8)
axes[0,1].set_title("Cosine Similarity Matrix
(1.0 = identical embeddings)", fontsize=11, fontweight='bold')

# Embedding distances (anomaly)
bar_colors = ['#E74C3C' if abs(z) > 1.5 else '#F39C12' if abs(z) > 1.0 else '#27AE60'
               for z in z_scores]
axes[1,0].bar([m[5:] for m in month_list], distances, color=bar_colors, edgecolor='white')
axes[1,0].axhline(mean_dist, color='black', linestyle='--', lw=1.5, label=f'Mean ({mean_dist:.3f})')
axes[1,0].axhline(mean_dist+1.5*std_dist, color='red', linestyle=':', lw=1.5, label='Anomaly threshold')
axes[1,0].set_ylabel("Distance from centroid")
axes[1,0].set_title("Embedding Distance — Anomaly Detection", fontsize=11, fontweight='bold')
axes[1,0].legend(fontsize=9)

# Embedding norms
norms = np.linalg.norm(emb_matrix, axis=1)
axes[1,1].plot([m[5:] for m in month_list], norms, 'b-D', lw=2.5, ms=8)
axes[1,1].set_ylabel("Embedding L2 Norm")
axes[1,1].set_title("Embedding Magnitude
(Seasonal variation pattern)", fontsize=11, fontweight='bold')
axes[1,1].grid(True, alpha=0.3)

plt.suptitle("DINOv3 Multi-Temporal Embedding Analysis — NYC 2024
"
             f"Model: dinov3_vitl16_sat | 1024-dim CLS embeddings",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(BASE_DIR/"embedding_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

## Summary

In [ ]:
print("=" * 60)
print("NOTEBOOK 24 — Multi-Temporal DINOv3 Embedding Analysis")
print("=" * 60)
print()
print(f"Scenes analysed : {len(embeddings)}")
print(f"Embedding dim   : 1024 (CLS token)")
print(f"PCA variance    : {pca.explained_variance_ratio_.sum():.1%} (PC1+PC2+PC3)")
print(f"Most similar    : {best_pair}  (r={max_sim:.4f})")
print(f"Least similar   : {worst_pair}  (r={min_sim:.4f})")
print(f"Anomalies found : {len(anomalies)}")
print()
print("Applications:")
print("  1. Rapid image retrieval from large archives")
print("  2. Flood/fire event detection (anomaly)")
print("  3. Scene classification without labels")
print("  4. Multi-temporal change narratives")
print()
print("Next: 25_vision_language_querying_clip_moondream.ipynb")